# Mining Subsidence Analysis Tool

## Overview
This notebook calculates and visualizes subsidence above coal mining panels using the **Influence Template Method**. 

### Key Features:
- **Pre-computed Influence Template**: 10 rings × 64 sectors = 640 elements per panel
- **Shapely Geometry**: Efficient polygon containment checking
- **Visualization**: Subsidence contours, panel boundaries, and influence patterns
- **Scalable**: Optimized for grids with thousands of evaluation points

### Theory:
The subsidence at any surface point is calculated by:
$$S_{total} = \sum_i (w_i \cdot S_{max})$$

where $w_i$ is the weighting factor for each influence element and $S_{max} = h \times e \times a$

## Section 1: Import Required Libraries

We'll use NumPy for numerical operations, Matplotlib for visualization, Shapely for geometric operations, and the math module for trigonometric functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from shapely.geometry import Point, Polygon
import math

## Section 2: Gather User Inputs

Define the mining parameters and panel geometry. You can modify these values or use `input()` to collect them interactively.

**Parameters:**
- **H**: Depth of cover (meters)
- **h**: Panel thickness (meters)
- **e**: Extraction ratio (0.7 for Bord & Pillar, 1.0 for Longwall)
- **a**: Subsidence factor (dimensionless, typically 0.3–0.8)
- **NEW**: NEW ratio = W_new / H (width of new panel relative to depth)
- **alpha**: Angle of draw (degrees)
- **grid_spacing**: Mesh grid spacing (meters)

In [ ]:
# Set default values for testing (you can replace with input() for interactive mode)

# Option 1: Use fixed values
H = 100.0  # Depth of cover (m)
h = 2.5    # Panel thickness (m)
e = 0.9    # Extraction ratio
a = 0.5    # Subsidence factor
NEW = 1.2  # NEW ratio
alpha_deg = 30.0  # Angle of draw (degrees)
grid_spacing = 5.0  # Mesh grid spacing (m)

# Option 2: Uncomment below for interactive input mode
# H = float(input("Enter Depth of Cover (H) in meters: "))
# h = float(input("Enter Panel Thickness (h) in meters: "))
# e = float(input("Enter Extraction Ratio (e.g., 0.7 for B&P, 1.0 for Longwall): "))
# a = float(input("Enter Subsidence Factor (a): "))
# NEW = float(input("Enter NEW ratio (W_new / H): "))
# alpha_deg = float(input("Enter Angle of draw (alpha) in degrees: "))
# grid_spacing = float(input("Enter mesh grid spacing (e.g., 5 or 10 meters): "))

# Convert angle to radians
alpha_rad = math.radians(alpha_deg)

# Calculate maximum subsidence
S_max = h * e * a

print(f"\n{'='*60}")
print(f"Mining Subsidence Analysis Parameters")
print(f"{'='*60}")
print(f"Depth of Cover (H):        {H:.2f} m")
print(f"Panel Thickness (h):       {h:.2f} m")
print(f"Extraction Ratio (e):      {e:.2f}")
print(f"Subsidence Factor (a):     {a:.2f}")
print(f"NEW Ratio:                 {NEW:.2f}")
print(f"Angle of Draw (α):         {alpha_deg:.2f}° ({alpha_rad:.4f} rad)")
print(f"Grid Spacing:              {grid_spacing:.2f} m")
print(f"Maximum Subsidence (S_max):{S_max:.4f} m")
print(f"{'='*60}\n")

## Section 3: Define Panel Geometry and Create Grid

Create the mining panel as a Shapely Polygon, calculate the influence radius R, and generate a 2D grid of evaluation points.

$$R = H \cdot \tan(\alpha)$$

The grid is expanded beyond the panel boundary by ±R on all sides to capture the full influence zone.

In [ ]:
# Define panel coordinates (easting, northing) - example rectangular panel
# You can modify these coordinates or input from external data
panel_coords = [
    (200, 100),   # Southwest corner
    (500, 100),   # Southeast corner
    (500, 400),   # Northeast corner
    (200, 400),   # Northwest corner
]

# Create Shapely Polygon
panel_poly = Polygon(panel_coords)

# Get bounding box
min_x, min_y, max_x, max_y = panel_poly.bounds

print(f"\nPanel Geometry:")
print(f"  Panel Bounds: X=[{min_x:.1f}, {max_x:.1f}], Y=[{min_y:.1f}, {max_y:.1f}]")
print(f"  Panel Area: {panel_poly.area:.0f} m²")

# Calculate influence radius
R = H * math.tan(alpha_rad)
print(f"  Influence Radius (R): {R:.2f} m")

# Expand grid bounds by R (the influence radius)
grid_min_x = min_x - R
grid_max_x = max_x + R
grid_min_y = min_y - R
grid_max_y = max_y + R

print(f"\nGrid Bounds (expanded by R):")
print(f"  X: [{grid_min_x:.1f}, {grid_max_x:.1f}]")
print(f"  Y: [{grid_min_y:.1f}, {grid_max_y:.1f}]")

# Create 1D arrays for X and Y
x_array = np.arange(grid_min_x, grid_max_x + grid_spacing, grid_spacing)
y_array = np.arange(grid_min_y, grid_max_y + grid_spacing, grid_spacing)

# Create 2D meshgrid
X_grid, Y_grid = np.meshgrid(x_array, y_array)

print(f"  Grid Points: {len(x_array)} × {len(y_array)} = {X_grid.size} points")
print()

## Section 4: Build the Influence Template

Pre-compute an influence template centered at (0, 0) with 10 rings and 64 sectors (total 640 elements).

**Template Structure:**
- **10 Rings**: Distributed radially from 0 to R
- **64 Sectors**: Distributed azimuthally around 360°
- **Each Element**: Stores (dx, dy, weight) for later translation

For ring $i$ and sector $j$:
$$r_{mid,i} = (i + 0.5) \times \frac{R}{10}$$
$$\phi_j = j \times \frac{2\pi}{64}$$
$$dx_j = r_{mid,i} \cdot \cos(\phi_j), \quad dy_j = r_{mid,i} \cdot \sin(\phi_j)$$

In [ ]:
# Build the influence template
num_rings = 10
num_sectors = 64
template_elements = []

# For each ring
for i in range(num_rings):
    # Calculate mid-radius for this ring
    r_mid = (i + 0.5) * (R / num_rings)
    
    # For each sector
    for j in range(num_sectors):
        # Calculate angle
        phi = j * (2 * np.pi / num_sectors)
        
        # Local coordinates relative to origin
        dx = r_mid * np.cos(phi)
        dy = r_mid * np.sin(phi)
        
        # Weighting: uniform weighting per sector
        # (Each ring and sector contributes equally: 1/(num_rings*num_sectors))
        weight = 1.0 / (num_rings * num_sectors)
        
        template_elements.append((dx, dy, weight))

print(f"Influence Template Built:")
print(f"  Total Elements: {len(template_elements)}")
print(f"  Rings: {num_rings}, Sectors: {num_sectors}")
print(f"  Weight per Element: {1.0 / len(template_elements):.6f}")
print(f"  Max Radius: {R:.2f} m\n")

## Section 5: Execute Main Calculation Engine

Loop through every grid point and apply the influence template. For each template element shifted to the grid point location, check if it falls within the panel using Shapely's `contains()` method.

**Algorithm:**
1. For each grid point $(X_p, Y_p)$:
2.     Initialize point_subsidence = 0
3.     For each template element (dx, dy, weight):
4.         Translate element: $(x_{elem}, y_{elem}) = (X_p + dx, Y_p + dy)$
5.         If panel contains $(x_{elem}, y_{elem})$:
6.             Accumulate: point_subsidence += weight × $S_{max}$
7.     Store result: $(X_p, Y_p, \text{point\_subsidence})$

In [ ]:
import time

# Main calculation: loop through all grid points
results = []
total_points = X_grid.size

print("Starting Subsidence Calculation...")
print(f"Evaluating {total_points} grid points...\n")

start_time = time.time()

# Flatten the grid arrays
X_flat = X_grid.flatten()
Y_flat = Y_grid.flatten()

# Process each grid point
for idx, (X_p, Y_p) in enumerate(zip(X_flat, Y_flat)):
    point_subsidence = 0.0
    
    # Check all template elements
    for dx, dy, weight in template_elements:
        # Translate template element to current grid point
        element_x = X_p + dx
        element_y = Y_p + dy
        
        # Create point at this location
        pt = Point(element_x, element_y)
        
        # Check if point is inside the panel
        if panel_poly.contains(pt):
            # This element contributes to subsidence
            point_subsidence += weight * S_max
    
    # Store result
    results.append((X_p, Y_p, point_subsidence))
    
    # Progress indicator
    if (idx + 1) % max(1, total_points // 10) == 0:
        elapsed = time.time() - start_time
        percent = 100 * (idx + 1) / total_points
        est_total = elapsed / ((idx + 1) / total_points)
        est_remaining = est_total - elapsed
        print(f"  {percent:5.1f}% ({idx+1:6d}/{total_points}) - "
              f"Elapsed: {elapsed:6.1f}s, Remaining: ~{est_remaining:6.1f}s")

elapsed_time = time.time() - start_time
print(f"\nCalculation Complete!")
print(f"  Total Time: {elapsed_time:.2f} seconds")
print(f"  Points/sec: {total_points/elapsed_time:.0f}\n")

## Section 6: Extract and Prepare Results Data

Convert 1D results list into 2D grids compatible with Matplotlib's contouring functions.

In [ ]:
# Extract results into separate arrays
X_results = np.array([r[0] for r in results])
Y_results = np.array([r[1] for r in results])
Z_results = np.array([r[2] for r in results])

# Reshape into 2D grids for plotting
X_2d = X_results.reshape(X_grid.shape)
Y_2d = Y_results.reshape(Y_grid.shape)
Z_2d = Z_results.reshape(X_grid.shape)

# Calculate statistics
max_subsidence = np.max(Z_results)
min_subsidence = np.min(Z_results)
mean_subsidence = np.mean(Z_results[Z_results > 0]) if np.any(Z_results > 0) else 0
non_zero_points = np.sum(Z_results > 0)

print("Subsidence Statistics:")
print(f"  Maximum Subsidence: {max_subsidence:.4f} m ({100*max_subsidence/S_max:.1f}% of S_max)")
print(f"  Minimum Subsidence: {min_subsidence:.4f} m")
print(f"  Mean (non-zero):    {mean_subsidence:.4f} m")
print(f"  Points with S > 0:  {non_zero_points} / {len(Z_results)}")
print()

## Section 7: Visualize Subsidence Trough and Influence Pattern

Create a comprehensive visualization showing:
1. **Panel boundary** (black outline)
2. **Filled contour plot** of subsidence using reversed colormap
3. **Grid boundary box** (dashed line)
4. **Influence circle diagram** with 10 concentric rings and 64 sectors

In [ ]:
# Create a comprehensive visualization
fig = plt.figure(figsize=(16, 12))

# ============================================================================
# PLOT 1: Subsidence Contour Map (Main plot, left side)
# ============================================================================
ax1 = plt.subplot(1, 2, 1)

# Create filled contour plot
levels = np.linspace(0, max_subsidence, 30)
cf = ax1.contourf(X_2d, Y_2d, Z_2d, levels=levels, cmap='viridis_r')

# Add contour lines
cs = ax1.contour(X_2d, Y_2d, Z_2d, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax1.clabel(cs, inline=True, fontsize=8, fmt='%.3f')

# Plot panel boundary
panel_x, panel_y = panel_poly.exterior.xy
ax1.plot(panel_x, panel_y, 'k-', linewidth=2, label='Panel Boundary')
ax1.fill(panel_x, panel_y, facecolor='lightgray', alpha=0.2, edgecolor='black')

# Plot grid boundary
ax1.plot([grid_min_x, grid_max_x, grid_max_x, grid_min_x, grid_min_x],
         [grid_min_y, grid_min_y, grid_max_y, grid_max_y, grid_min_y],
         'r--', linewidth=1.5, alpha=0.7, label='Grid Boundary')

# Colorbar
cbar = plt.colorbar(cf, ax=ax1, label='Subsidence (m)', pad=0.02)

ax1.set_xlabel('Easting (m)', fontsize=11)
ax1.set_ylabel('Northing (m)', fontsize=11)
ax1.set_title('Mining Subsidence Trough\n(Filled Contours)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.2)
ax1.legend(loc='upper right')
ax1.set_aspect('equal')

# ============================================================================
# PLOT 2: Influence Template Diagram (right side)
# ============================================================================
ax2 = plt.subplot(1, 2, 2)

# Choose a reference grid point near the panel center
ref_x = (panel_coords[0][0] + panel_coords[2][0]) / 2
ref_y = (panel_coords[0][1] + panel_coords[2][1]) / 2

# Draw concentric rings
for i in range(1, num_rings + 1):
    r_outer = i * (R / num_rings)
    circle = Circle((ref_x, ref_y), r_outer, fill=False, 
                    edgecolor='blue', alpha=0.3, linestyle='--', linewidth=1)
    ax2.add_patch(circle)

# Draw radial sectors (every 8 sectors to avoid clutter = 360/8 = 45 degrees)
sector_step = 8
for j in range(0, num_sectors, sector_step):
    phi = j * (2 * np.pi / num_sectors)
    x_end = ref_x + R * np.cos(phi)
    y_end = ref_y + R * np.sin(phi)
    ax2.plot([ref_x, x_end], [ref_y, y_end], 'b-', alpha=0.3, linewidth=0.8)

# Plot reference point
ax2.plot(ref_x, ref_y, 'r*', markersize=15, label='Reference Point')

# Plot panel
ax2.plot(panel_x, panel_y, 'k-', linewidth=2, label='Panel Boundary')
ax2.fill(panel_x, panel_y, facecolor='lightgray', alpha=0.2)

# Add influence circle (maximum extent)
circle_influence = Circle((ref_x, ref_y), R, fill=False, 
                          edgecolor='red', linewidth=2, label=f'Influence Radius R={R:.0f}m')
ax2.add_patch(circle_influence)

ax2.set_xlabel('Easting (m)', fontsize=11)
ax2.set_ylabel('Northing (m)', fontsize=11)
ax2.set_title('Influence Template Visualization\n(10 Rings × 64 Sectors = 640 Elements)', 
              fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.2)
ax2.legend(loc='upper right')
ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('/workspaces/Subsidence/subsidence_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to: subsidence_visualization.png")

## Summary & Results Interpretation

### Key Outputs:
- **S_max**: Maximum possible subsidence = h × e × a
- **Influence Radius (R)**: H × tan(α) — the zone where the panel affects surface
- **Grid Points**: Total evaluation locations for subsidence calculation
- **Calculation Speed**: Demonstrates the efficiency of the pre-computed template method

### Output Files:
- **subsidence_visualization.png**: Main visualization showing contours and influence pattern

### How to Use This Notebook:
1. **Modify Panel Coordinates**: Edit `panel_coords` to define your mining panel geometry
2. **Adjust Parameters**: Change H, h, e, a, NEW, alpha_deg, grid_spacing as needed
3. **Switch to Interactive Mode**: Uncomment the `input()` calls in Section 2 for terminal input
4. **Run All Cells**: Execute the entire notebook to see results
5. **Export Results**: Use numpy/pandas to save subsidence data to CSV/Excel if needed

### Advanced Extensions:
- Implement the **Modified Method** with distance-based weighting
- Add multiple panel support (loop template over multiple panels)
- Export 3D data for GIS visualization
- Implement probabilistic approaches
- Add validation against field measurements

### References:
- Influence Function Method: Widely used in mining subsidence prediction
- NEW Ratio: Newer and Kratzsch Method
- Angle of Draw: 20°–40° typical for coal mining

## Optional: Export Results to CSV and Create Additional Plots

In [ ]:
# Export results to CSV format
import csv

output_file = '/workspaces/Subsidence/subsidence_results.csv'

with open(output_file, 'w', newline='') as f:
    writer = csv.writer(f)
    # Write header
    writer.writerow(['Easting (m)', 'Northing (m)', 'Subsidence (m)'])
    # Write data rows
    for x, y, z in results:
        writer.writerow([f'{x:.2f}', f'{y:.2f}', f'{z:.6f}'])

print(f"Results exported to: {output_file}")
print(f"Total records: {len(results)}")

# Create a 3D surface plot
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot surface
surf = ax.plot_surface(X_2d, Y_2d, Z_2d, cmap='viridis_r', alpha=0.8, edgecolor='none')

# Add panel outline at base
panel_x_arr, panel_y_arr = np.array(panel_x), np.array(panel_y)
panel_z = np.zeros_like(panel_x_arr)
ax.plot(panel_x_arr, panel_y_arr, panel_z, 'k-', linewidth=2)

ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.set_zlabel('Subsidence (m)')
ax.set_title('3D Subsidence Surface', fontweight='bold')
cbar = fig.colorbar(surf, ax=ax, pad=0.1, label='Subsidence (m)')

plt.savefig('/workspaces/Subsidence/subsidence_3d_surface.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n3D surface plot saved to: subsidence_3d_surface.png")